# Adaptive Sleep Replay Lab

Version-controlled Colab entry point for the Split-MNIST continual-learning pilots. This notebook runs the repository scripts; it is **not** a reproduction of an LLM sleep/dreaming paper. The primary question is replay scheduling. MoE is an architecture sidecar.

Open this notebook from the repository in Colab, choose **Runtime → Change runtime type → T4 GPU**, then run the setup and the experiment cells you need. The five-seed cells take a few minutes. Outputs are saved under `artifacts/` in the Colab clone; download or commit them only after inspecting the generated files.

Repository source: [adaptive-sleep-replay](https://github.com/dakshdoesdev/adaptive-sleep-replay). The original Colab notebook was not present in the repository or local project when this notebook was added, so this is a clean tracked lab notebook rather than an overwrite of that Drive file.

In [ ]:
from pathlib import Path
import subprocess

REPO = Path('/content/adaptive-sleep-replay-lab')
if not REPO.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/dakshdoesdev/adaptive-sleep-replay.git', str(REPO)
    ], check=True)
print('Project directory:', REPO)
print(subprocess.run(
    ['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO,
    check=True, capture_output=True, text=True
).stdout.strip())

In [ ]:
%cd /content/adaptive-sleep-replay-lab
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before running experiments.'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
!python -m unittest discover -s experiments -v

## Phase 1 — sequential vs. uniform replay

Historical pilots are documented in [`artifacts/RESULTS.md`](../artifacts/RESULTS.md). This rerun is exploratory and adds replay training work.

In [ ]:
import json
from pathlib import Path
import subprocess

out = 'artifacts/notebook_phase_01_t4.json'
plot = 'artifacts/notebook_phase_01_t4.png'
with open('/tmp/notebook_phase_01.log', 'w') as log:
    subprocess.run([
        'python', 'experiments/split_mnist_baseline.py',
        '--epochs', '3', '--seeds', '42', '43', '44', '45', '46',
        '--out', out, '--plot', plot
    ], check=True, stdout=log)
phase1 = json.loads(Path(out).read_text())
for method, values in phase1['methods'].items():
    print(method, 'final accuracy:', round(values['final_average_accuracy'] * 100, 2),
          '+/-', round(values['final_average_accuracy_std'] * 100, 2),
          '| forgetting:', round(values['average_forgetting'] * 100, 2), 'pp')

## Phase 2 — periodic vs. random replay timing

Both schedules use the same replay count and burst length. See [`artifacts/PHASE_02_RESULTS.md`](../artifacts/PHASE_02_RESULTS.md) for the published pilot and caveats.

In [ ]:
out = 'artifacts/notebook_phase_02_t4.json'
plot = 'artifacts/notebook_phase_02_t4.png'
with open('/tmp/notebook_phase_02.log', 'w') as log:
    subprocess.run([
        'python', 'experiments/replay_burst_scheduling.py',
        '--epochs', '3', '--seeds', '42', '43', '44', '45', '46',
        '--out', out, '--plot', plot
    ], check=True, stdout=log)
phase2 = json.loads(Path(out).read_text())
for method, values in phase2['methods'].items():
    print(method, 'final accuracy:', round(values['final_average_accuracy'] * 100, 2),
          '| forgetting:', round(values['average_forgetting'] * 100, 2), 'pp',
          '| replay examples:', values['replay_examples'])
print('random - periodic paired accuracy (pp):', round(
    phase2['paired_differences']['final_average_accuracy']['mean'] * 100, 2
))

## Phase 3 diagnostic — active-compute-matched dense vs. sparse MoE

This corrected comparison uses 128-wide experts, matching estimated active forward MACs within 1% while allowing the MoE more total parameters. Four within-task diagnostic windows record routing, router probabilities, classification loss, and balance-loss strength. The original 32-wide MoE result is retained as historical pilot data but is not an apples-to-apples active-capacity comparison. Protocol: [`experiments/phase-03-moe-diagnostic.md`](../experiments/phase-03-moe-diagnostic.md).

### Recorded diagnostic run (local RTX 3050)

Five paired seeds finished on the local RTX 3050: dense accuracy **43.95 ± 1.71%**, MoE **29.08 ± 2.24%**; MoE-minus-dense paired difference **−14.87 pp**. Active forward MAC estimates matched within 0.38%, but the router collapsed rapidly after task 1. Full per-window signals and caveats: [`artifacts/PHASE_03_DIAGNOSTIC_RESULTS.md`](../artifacts/PHASE_03_DIAGNOSTIC_RESULTS.md). This stored run used source commit `56b2701`; rerunning this notebook on a T4 will create a distinct run.

In [ ]:
out = 'artifacts/notebook_phase_03_moe_compute_matched_t4.json'
plot = 'artifacts/notebook_phase_03_moe_compute_matched_t4.png'
with open('/tmp/notebook_phase_03.log', 'w') as log:
    subprocess.run([
        'python', 'experiments/moe_split_mnist.py',
        '--moe-bottleneck', '128', '--epochs', '3',
        '--seeds', '42', '43', '44', '45', '46',
        '--out', out, '--plot', plot
    ], check=True, stdout=log)
phase3 = json.loads(Path(out).read_text())
for method, values in phase3['methods'].items():
    profile = values['model_profile']
    print(method, 'final accuracy:', round(values['final_average_accuracy'] * 100, 2),
          '| forgetting:', round(values['average_forgetting'] * 100, 2), 'pp',
          '| total params:', profile['total_parameters'],
          '| active MACs/example:', profile['estimated_forward_macs_per_example'])
print('MoE - dense paired accuracy (pp):', round(
    phase3['paired_differences']['final_average_accuracy']['mean'] * 100, 2
))
print('MoE task-1 routing by training window:')
for window in phase3['runs']['42']['methods']['sparse_moe']['incoming_router_diagnostics_by_task'][1]['windows']:
    print('window', window['window'], 'routes', window['route_fraction'],
          'weighted balance/classification loss ratio',
          window['weighted_auxiliary_to_classification_loss_ratio'])

## Interpretation boundary

These are small Split-MNIST pilots, not final thesis evidence and not LLM sleep/dreaming reproductions. Do not tune from the final test metrics. The corrected MoE comparison matches estimated active MACs, not total parameter count or wall-clock compute. Review the JSON artifacts and protocol before making claims.